In [ ]:

# If needed, uncomment:
# !pip install -q pandas matplotlib seaborn plotly

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns

warnings.filterwarnings("ignore")

# ================================================================
# 1. FILE PATH IN COLAB
# ================================================================
file_path = "Table1.csv"


def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ================================================================
# 2. SHARED PREPARATION
# ================================================================
def prepare_superuser_base(df: pd.DataFrame):
    col_id = "Record ID"
    col_ciap = next((c for c in df.columns if "ciap" in c.lower() or "diagnostico" in c.lower()), None)
    col_status = next((c for c in df.columns if any(p in c.lower() for p in ["desfecho", "status"])), None)

    if col_ciap is None:
        raise ValueError("No CIAP/diagnosis column was found.")
    if col_status is None:
        raise ValueError("No status/outcome column was found.")

    df_attended = df[
        ~df[col_status].astype(str).str.contains("não compareceu|falta", case=False, na=False)
    ].copy()

    profile = df_attended.groupby(col_id).size().reset_index(name="Total_Consultations")
    p95_threshold = profile["Total_Consultations"].quantile(0.95)

    super_ids = profile[profile["Total_Consultations"] >= p95_threshold][col_id]
    df_super = df_attended[df_attended[col_id].isin(super_ids)].copy()

    ciap_names = {
        "A": "General / Non-specific",
        "D": "Digestive",
        "K": "Circulatory",
        "L": "Musculoskeletal",
        "P": "Psychological",
        "R": "Respiratory",
        "S": "Skin",
        "T": "Endocrine / Metabolic",
        "B": "Blood",
    }

    df_super["CIAP_Chapter"] = df_super[col_ciap].apply(
        lambda x: ciap_names.get(str(x)[0].upper(), "Other / Not Informed")
    )

    return profile, p95_threshold, super_ids, df_super


# ================================================================
# 3. FIGURE 1 — USAGE INTENSITY + CLINICAL PROFILE
# ================================================================
def figure_1_superuser_profile(df: pd.DataFrame) -> None:
    profile, p95_threshold, super_ids, df_super = prepare_superuser_base(df)

    plt.figure(figsize=(15, 6))

    # ------------------------------------------------------------
    # Chart 1 — Histogram of service use intensity
    # ------------------------------------------------------------
    plt.subplot(1, 2, 1)
    sns.histplot(profile["Total_Consultations"], bins=20, kde=True, color="#2c3e50")
    plt.axvline(
        p95_threshold,
        color="#e74c3c",
        linestyle="--",
        label=f"P95 Threshold (>= {p95_threshold:.0f} consultations)",
    )
    plt.title("Distribution of Service Use Intensity", fontweight="bold")
    plt.xlabel("Consultations per Patient")
    plt.ylabel("Number of Patients")
    plt.legend()

    # ------------------------------------------------------------
    # Chart 2 — Clinical profile of superusers
    # ------------------------------------------------------------
    plt.subplot(1, 2, 2)
    ciap_ranking = df_super["CIAP_Chapter"].value_counts().head(8)
    sns.barplot(x=ciap_ranking.values, y=ciap_ranking.index, palette="viridis")
    plt.title(
        f"Main Demands of Superusers\n(N={len(super_ids)} patients)",
        fontweight="bold",
    )
    plt.xlabel("Total Accumulated Consultations")
    plt.ylabel("")

    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Export article support table
    # ------------------------------------------------------------
    article_table = ciap_ranking.reset_index()
    article_table.columns = ["CIAP Chapter", "Total Consultations"]
    article_table["Proportion (%)"] = (
        article_table["Total Consultations"] / len(df_super) * 100
    ).round(1)

    output_file = "Superusers_Profile_Table_EN.csv"
    article_table.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 60)
    print("SUPPORT TABLE: SUPERUSER PROFILE")
    print("=" * 60)
    print(article_table.to_string(index=False))
    print("-" * 60)
    print(f"Hyperutilization threshold (P95): >= {p95_threshold:.0f} consultations per patient.")
    print(f"Exported table: '{output_file}'")
    print("=" * 60 + "\n")


# ================================================================
# 4. FIGURE 2 — SANKEY OF HYPERUTILIZATION FLOW
# ================================================================
def figure_2_superuser_sankey(df: pd.DataFrame) -> None:
    _, p95_threshold, super_ids, df_super = prepare_superuser_base(df)

    ciap_distribution = df_super["CIAP_Chapter"].value_counts()

    n_patients = len(super_ids)
    n_consultations = len(df_super)

    base_links = []
    base_links.append(["Superusers (P95)", "Total Consultations", n_consultations])

    for ciap, value in ciap_distribution.items():
        base_links.append(["Total Consultations", ciap, value])

    sankey_base = pd.DataFrame(base_links, columns=["Source", "Target", "Value"])

    all_nodes = pd.unique(sankey_base[["Source", "Target"]].values.ravel("K"))
    mapping = {node: i for i, node in enumerate(all_nodes)}

    sankey_base["source_idx"] = sankey_base["Source"].map(mapping)
    sankey_base["target_idx"] = sankey_base["Target"].map(mapping)

    display_labels = []
    for node in all_nodes:
        if node == "Superusers (P95)":
            display_labels.append(f"SUPERUSERS<br>(N={n_patients} Individuals)")
        elif node == "Total Consultations":
            display_labels.append(f"CONSULTATION LOAD<br>(N={n_consultations} Visits)")
        else:
            v = ciap_distribution.get(node, 0)
            p = (v / n_consultations * 100) if n_consultations > 0 else 0
            display_labels.append(f"{node}<br>({v} consultations | {p:.1f}%)")

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=20,
                    thickness=20,
                    line=dict(color="black", width=0.5),
                    label=display_labels,
                    color="#e67e22",
                ),
                link=dict(
                    source=sankey_base["source_idx"],
                    target=sankey_base["target_idx"],
                    value=sankey_base["Value"],
                    color="rgba(230, 126, 34, 0.3)",
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="Hyperutilization Flow: Clinical Distribution of Superusers (Top 5%)",
        font_size=11,
    )
    fig.show()

    output_file = "Hyperutilization_Sankey_Base_EN.csv"
    sankey_base[["Source", "Target", "Value"]].to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 70)
    print("SANKEY DIAGRAM BASE (SOURCE -> TARGET)")
    print("=" * 70)
    print(sankey_base[["Source", "Target", "Value"]].to_string(index=False))
    print("-" * 70)
    print(f"Hyperutilization threshold (P95): >= {p95_threshold:.0f} consultations per patient.")
    print(f"Exported table: '{output_file}'")
    print("=" * 70 + "\n")


# ================================================================
# 5. RUN EVERYTHING
# ================================================================
def main():
    df = load_csv()
    figure_1_superuser_profile(df)
    figure_2_superuser_sankey(df)


main()